# NIFTY 50 Financial Forecast Model

## Week 2 — Creating Financial Forecasts

**Student:** Pavan Poriya

---

## 1. Week 2 Objective

The objective of Week 2 is to build a basic financial forecast model for a chosen financial indicator using the NIFTY 50 historical dataset analyzed in Week 1.

This notebook covers:
- Selecting a financial indicator for forecasting
- Engineering lag-based features from historical data
- Implementing a supervised regression model for time-series forecasting
- Evaluating model performance on unseen chronological data
- Generating a 30-session future forecast
- Visualizing and interpreting results

**Important:** This notebook uses the same dataset from Week 1. The Week 1 notebook and all its outputs remain unchanged.

## 2. Dataset Loading

We load the same NIFTY 50 historical CSV file used in Week 1. The dataset contains daily OHLCV (Open, High, Low, Close, Volume) data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
df = pd.read_csv("DataSet/NIFTY50_1995_to_Feb_2026.csv")
print(f"Raw dataset shape: {df.shape}")
df.head()

Raw dataset shape: (4523, 10)


,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
0,NaN,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,NaN,NaN,NaN,NaN
1,2007-09-17,4494.64990234375,4549.0498046875,4482.85009765625,4518.4501953125,0,^NSEI,NaN,NaN,NaN
2,2007-09-18,4546.2001953125,4551.7998046875,4481.5498046875,4494.10009765625,0,^NSEI,1.146926,NaN,NaN
3,2007-09-19,4732.35009765625,4739.0,4550.25,4550.25,0,^NSEI,4.094626,NaN,NaN
4,2007-09-20,4747.5498046875,4760.85009765625,4721.14990234375,4734.85009765625,0,^NSEI,0.321187,NaN,NaN


## 3. Data Preparation

We replicate the exact data cleaning steps from Week 1:
1. Remove the metadata row (index 0)
2. Convert `Date` to datetime
3. Convert numerical columns to appropriate types
4. Drop rows where `Close` is missing
5. Sort by date to ensure chronological order

In [3]:
# Remove metadata row (replicating Week 1 cleaning)
df = df.drop(index=0).reset_index(drop=True)

# Convert types
df["Date"] = pd.to_datetime(df["Date"])
df["Close"] = pd.to_numeric(df["Close"])
df["High"] = pd.to_numeric(df["High"])
df["Low"] = pd.to_numeric(df["Low"])
df["Open"] = pd.to_numeric(df["Open"])
df["Volume"] = pd.to_numeric(df["Volume"])

# Drop rows where Close is NaN
df = df.dropna(subset=["Close"]).reset_index(drop=True)

# Ensure sorted by date
df = df.sort_values("Date").reset_index(drop=True)

print(f"Cleaned dataset shape: {df.shape}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Total trading days: {len(df)}")

Cleaned dataset shape: (4522, 10)
Date range: 2007-09-17 to 2026-02-20
Total trading days: 4522


In [4]:
df[["Date", "Close", "High", "Low", "Open", "Volume"]].describe()

,Date,Close,High,Low,Open,Volume
count,4522,4522.000000,4522.000000,4522.000000,4522.000000,4.522000e+03
mean,2016-12-12 19:25:30.119416,11028.401908,11092.470576,10961.462004,11034.684886,2.124643e+05
min,2007-09-17 00:00:00,2524.199951,2585.300049,2252.750000,2553.600098,0.000000e+00
25%,2012-05-02 06:00:00,5691.074829,5726.737549,5650.674805,5693.312378,0.000000e+00
50%,2016-12-22 12:00:00,8744.149902,8792.049805,8703.550293,8756.625000,1.898000e+05
75%,2021-07-27 18:00:00,15788.375244,15863.100342,15720.899902,15799.562256,2.956750e+05
max,2026-02-20 00:00:00,26328.550781,26373.199219,26210.050781,26333.699219,1.811000e+06
std,NaN,6441.130396,6462.049908,6418.805216,6443.479213,2.053608e+05


## 4. Indicator Selection

**Selected Indicator: NIFTY 50 Closing Price (`Close`)**

**Justification:**
- The closing price is the most widely used price indicator in financial markets.
- It represents the final consensus price at the end of each trading session.
- It was already the primary variable analyzed in Week 1.
- Forecasting closing prices is a fundamental task in financial modeling.
- The closing price series is continuous, numeric, and has sufficient historical depth for time-series modeling.

## 5. Feature Engineering

We create **lag-based features** from the historical closing price. Each feature uses only information available **before** the target date to prevent future data leakage.

**Features created:**

| Feature | Description |
|---------|-------------|
| `Close_Lag_1` | Closing price 1 day ago |
| `Close_Lag_5` | Closing price 5 days ago |
| `Close_Lag_10` | Closing price 10 days ago |
| `Close_Lag_20` | Closing price 20 days ago |
| `MA_5` | 5-day simple moving average of Close |
| `MA_20` | 20-day simple moving average of Close |
| `Daily_Return_Pct` | Daily percentage return |

**Why these features?**
- **Lag features** capture short-term and medium-term price momentum.
- **Moving averages** smooth out daily noise and capture underlying trends.
- **Daily return** captures the most recent price change direction.
- All features are strictly historical — no future information is used.

In [5]:
# Create lag features from historical Close prices
df["Close_Lag_1"] = df["Close"].shift(1)
df["Close_Lag_5"] = df["Close"].shift(5)
df["Close_Lag_10"] = df["Close"].shift(10)
df["Close_Lag_20"] = df["Close"].shift(20)

# Create moving average features
df["MA_5"] = df["Close"].rolling(window=5).mean()
df["MA_20"] = df["Close"].rolling(window=20).mean()

# Calculate daily return percentage from Close prices
df["Daily_Return_Pct"] = df["Close"].pct_change() * 100

print("Feature engineering complete.")
print(f"Features created: Close_Lag_1, Close_Lag_5, Close_Lag_10, Close_Lag_20, MA_5, MA_20, Daily_Return_Pct")

Feature engineering complete.
Features created: Close_Lag_1, Close_Lag_5, Close_Lag_10, Close_Lag_20, MA_5, MA_20, Daily_Return_Pct


In [6]:
# Define feature columns
feature_columns = ["Close_Lag_1", "Close_Lag_5", "Close_Lag_10", "Close_Lag_20", "MA_5", "MA_20", "Daily_Return_Pct"]
target_column = "Close"

# Drop rows with NaN in features or target
df_model = df.dropna(subset=feature_columns + [target_column]).copy()
df_model = df_model.sort_values("Date").reset_index(drop=True)

print(f"Rows available for modeling: {len(df_model)}")
print(f"Date range for modeling: {df_model['Date'].min().date()} to {df_model['Date'].max().date()}")
print(f"\nMissing values in features:")
print(df_model[feature_columns].isnull().sum())

Rows available for modeling: 4502
Date range for modeling: 2007-10-16 to 2026-02-20

Missing values in features:
Close_Lag_1         0
Close_Lag_5         0
Close_Lag_10        0
Close_Lag_20        0
MA_5                0
MA_20               0
Daily_Return_Pct    0
dtype: int64


In [7]:
df_model[["Date", "Close"] + feature_columns].head(10)

,Date,Close,Close_Lag_1,Close_Lag_5,Close_Lag_10,Close_Lag_20,MA_5,MA_20,Daily_Return_Pct
0,2007-10-16,5668.049805,5670.399902,5327.250000,5068.950195,4494.649902,5546.600000,5125.835010,-0.041445
1,2007-10-17,5559.299805,5668.049805,5441.450195,5210.799805,4546.200195,5570.169922,5176.489990,-1.918649
2,2007-10-18,5351.000000,5559.299805,5524.850098,5208.649902,4732.350098,5535.399902,5207.422485,-3.746871
3,2007-10-19,5215.299805,5351.000000,5428.250000,5185.850098,4747.549805,5492.809863,5230.809985,-2.535978
4,2007-10-22,5184.000000,5215.299805,5670.399902,5085.100098,4837.549805,5395.529883,5248.132495,-0.600154
5,2007-10-23,5473.700195,5184.000000,5668.049805,5327.250000,4932.200195,5356.659961,5275.207495,5.588353
6,2007-10-24,5496.149902,5473.700195,5559.299805,5441.450195,4938.850098,5344.029980,5303.072485,0.410138
7,2007-10-25,5568.950195,5496.149902,5351.000000,5524.850098,4940.500000,5387.620020,5334.494995,1.324569
8,2007-10-26,5702.299805,5568.950195,5215.299805,5428.250000,5000.549805,5485.020020,5369.582495,2.394520
9,2007-10-29,5905.899902,5702.299805,5184.000000,5670.399902,5021.350098,5629.400000,5413.809985,3.570491


## 6. Train/Test Split

**Critical: Chronological Split**

Since this is financial time-series data, we **must not** randomly shuffle the data. Random shuffling would allow future information to leak into the training set, producing overly optimistic results.

**Split strategy:**
- **Training set:** First 80% of the data (chronologically earlier)
- **Testing set:** Last 20% of the data (chronologically later)

This ensures:
- The model is trained only on past observations
- The test set represents truly unseen future data
- The evaluation simulates a real forecasting scenario

In [8]:
# Define features and target
X = df_model[feature_columns].values
y = df_model[target_column].values
dates = df_model["Date"].values

# Chronological 80/20 split
split_index = int(len(X) * 0.80)

X_train = X[:split_index]
X_test = X[split_index:]
y_train = y[:split_index]
y_test = y[split_index:]
dates_train = dates[:split_index]
dates_test = dates[split_index:]

print(f"Training set: {len(X_train)} samples")
print(f"  Date range: {pd.Timestamp(dates_train[0]).date()} to {pd.Timestamp(dates_train[-1]).date()}")
print(f"Testing set:  {len(X_test)} samples")
print(f"  Date range: {pd.Timestamp(dates_test[0]).date()} to {pd.Timestamp(dates_test[-1]).date()}")
print(f"\nSplit ratio: {len(X_train)/len(X)*100:.1f}% train / {len(X_test)/len(X)*100:.1f}% test")

Training set: 3601 samples
  Date range: 2007-10-16 to 2022-06-29
Testing set:  901 samples
  Date range: 2022-06-30 to 2026-02-20

Split ratio: 80.0% train / 20.0% test


## 7. Model Design

**Model: Linear Regression**

We use **Linear Regression** as our baseline forecasting model. This is appropriate for Week 2 because:

1. **Simplicity:** Linear Regression is easy to understand, implement, and explain.
2. **Interpretability:** The model coefficients show the relationship between each lag feature and the predicted closing price.
3. **Reproducibility:** The results can be exactly reproduced.
4. **Baseline purpose:** It serves as a strong baseline against which more complex models could be compared.
5. **No overfitting risk:** With only 7 features, the model is unlikely to overfit significantly.

**Model equation:**

`Close = β₀ + β₁·Close_Lag_1 + β₂·Close_Lag_5 + β₃·Close_Lag_10 + β₄·Close_Lag_20 + β₅·MA_5 + β₆·MA_20 + β₇·Daily_Return_Pct`

## 8. Model Training

In [9]:
# Initialize and train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully.")
print(f"\nModel coefficients:")
for name, coef in zip(feature_columns, model.coef_):
    print(f"  {name:20s}: {coef:12.6f}")
print(f"\nIntercept: {model.intercept_:.6f}")

Model trained successfully.

Model coefficients:
  Close_Lag_1         :     0.837639
  Close_Lag_5         :    -0.075091
  Close_Lag_10        :     0.000978
  Close_Lag_20        :     0.001212
  MA_5                :     0.239807
  MA_20               :    -0.004286
  Daily_Return_Pct    :    62.500278

Intercept: -1.925680


## 9. Model Evaluation

We evaluate the model on the **unseen chronological test set** using the following metrics:

| Metric | Description |
|--------|-------------|
| **MAE** (Mean Absolute Error) | Average absolute difference between actual and predicted prices |
| **RMSE** (Root Mean Squared Error) | Square root of average squared errors; penalizes large errors more |
| **MAPE** (Mean Absolute Percentage Error) | Average absolute percentage error; intuitive for comparison |
| **R²** (Coefficient of Determination) | Proportion of variance explained by the model |

**Note:** All metrics are calculated from actual model predictions — no values are manually entered.

In [10]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
r2 = r2_score(y_test, y_pred)

print("=" * 55)
print("         MODEL EVALUATION METRICS (Test Set)")
print("=" * 55)
print(f"  MAE  (Mean Absolute Error):      {mae:.2f}")
print(f"  RMSE (Root Mean Squared Error):   {rmse:.2f}")
print(f"  MAPE (Mean Absolute % Error):     {mape:.4f}%")
print(f"  R²   (Coefficient of Determination): {r2:.6f}")
print("=" * 55)

         MODEL EVALUATION METRICS (Test Set)
  MAE  (Mean Absolute Error):      82.01
  RMSE (Root Mean Squared Error):   114.23
  MAPE (Mean Absolute % Error):     0.3747%
  R²   (Coefficient of Determination): 0.998622


In [11]:
# Interpretation of metrics
print("METRIC INTERPRETATION:")
print(f"\n• MAE = {mae:.2f}: On average, the model's prediction is off by approximately {mae:.2f} NIFTY points.")
print(f"• RMSE = {rmse:.2f}: The typical prediction error (penalizing larger errors) is about {rmse:.2f} points.")
print(f"• MAPE = {mape:.4f}%: The model's average prediction error is about {mape:.4f}% of the actual price.")
print(f"• R² = {r2:.6f}: The model explains approximately {r2*100:.2f}% of the variance in the test set closing prices.")

METRIC INTERPRETATION:

• MAE = 82.01: On average, the model's prediction is off by approximately 82.01 NIFTY points.
• RMSE = 114.23: The typical prediction error (penalizing larger errors) is about 114.23 points.
• MAPE = 0.3747%: The model's average prediction error is about 0.3747% of the actual price.
• R² = 0.998622: The model explains approximately 99.86% of the variance in the test set closing prices.


## 10. Actual vs Predicted Prices

We visualize the model's predictions against the actual closing prices over the test period.

In [12]:
# Create Actual vs Predicted chart
fig, ax = plt.subplots(figsize=(14, 6))

test_dates = pd.to_datetime(dates_test)

ax.plot(test_dates, y_test, label="Actual Close", color="#1f77b4", linewidth=1.5)
ax.plot(test_dates, y_pred, label="Predicted Close", color="#ff7f0e", linewidth=1.5, linestyle="--")

ax.set_title("NIFTY 50 — Actual vs Predicted Closing Price (Test Set)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Closing Price (₹)", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("assets/week2_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: assets/week2_actual_vs_predicted.png")

Chart saved: assets/week2_actual_vs_predicted.png


In [13]:
# Prediction error analysis
errors = y_test - y_pred
abs_errors = np.abs(errors)

print("PREDICTION ERROR ANALYSIS:")
print(f"  Mean error (bias):     {np.mean(errors):.2f}")
print(f"  Median absolute error: {np.median(abs_errors):.2f}")
print(f"  Max absolute error:    {np.max(abs_errors):.2f}")
print(f"  Min absolute error:    {np.min(abs_errors):.2f}")
print(f"  Std of errors:         {np.std(errors):.2f}")
print(f"  Days with overprediction: {(errors > 0).sum()} / {len(errors)}")
print(f"  Days with underprediction: {(errors < 0).sum()} / {len(errors)}")

PREDICTION ERROR ANALYSIS:
  Mean error (bias):     2.83
  Median absolute error: 61.59
  Max absolute error:    877.43
  Min absolute error:    0.00
  Std of errors:         114.20
  Days with overprediction: 481 / 901
  Days with underprediction: 420 / 901


## 11. Forecasting Future Values

We generate a **30-session future forecast** using recursive multi-step forecasting.

**How recursive forecasting works:**
1. We start with the last known data point from the dataset.
2. We predict the next day's closing price using lag features computed from historical + previously predicted values.
3. We then use that predicted value as input for the next day's forecast.
4. This process repeats for 30 trading sessions.

**Important caveats:**
- Recursive forecasting can accumulate errors over time.
- The forecast becomes increasingly uncertain further into the future.
- This is a **model-based, indicative forecast** — not a guarantee of future market behavior.

In [14]:
# Build a historical Close series for recursive forecasting
close_series = df_model["Close"].values.tolist()
last_known_close = close_series[-1]

print(f"Last observed Close price: ₹{last_known_close:.2f}")
print(f"Last observed date: {pd.Timestamp(dates[-1]).date()}")
print(f"Forecast horizon: 30 trading sessions")

Last observed Close price: ₹25571.25
Last observed date: 2026-02-20
Forecast horizon: 30 trading sessions


In [15]:
# Recursive 30-session future forecast
forecast_horizon = 30
future_forecasts = []
temp_close = list(close_series)  # copy of historical close prices

for i in range(forecast_horizon):
    # Compute features from the current state of temp_close
    n = len(temp_close)
    
    close_lag_1 = temp_close[-1]
    close_lag_5 = temp_close[-5] if n >= 5 else temp_close[0]
    close_lag_10 = temp_close[-10] if n >= 10 else temp_close[0]
    close_lag_20 = temp_close[-20] if n >= 20 else temp_close[0]
    
    ma_5 = np.mean(temp_close[-5:]) if n >= 5 else np.mean(temp_close)
    ma_20 = np.mean(temp_close[-20:]) if n >= 20 else np.mean(temp_close)
    
    daily_return = ((temp_close[-1] - temp_close[-2]) / temp_close[-2]) * 100 if n >= 2 else 0.0
    
    # Create feature vector
    features = np.array([[close_lag_1, close_lag_5, close_lag_10, close_lag_20, ma_5, ma_20, daily_return]])
    
    # Predict
    predicted_close = model.predict(features)[0]
    
    # Store forecast
    future_forecasts.append(predicted_close)
    
    # Append prediction to temp_close for next iteration
    temp_close.append(predicted_close)

# Generate future trading dates (skip weekends)
last_date = pd.Timestamp(dates[-1])
future_dates = []
current_date = last_date
count = 0
while count < forecast_horizon:
    current_date += pd.Timedelta(days=1)
    if current_date.weekday() < 5:  # Monday=0 to Friday=4
        future_dates.append(current_date)
        count += 1

future_dates = np.array(future_dates)

print(f"Future forecast generated: {len(future_forecasts)} sessions")
print(f"Forecast period: {future_dates[0].date()} to {future_dates[-1].date()}")

Future forecast generated: 30 sessions
Forecast period: 2026-02-23 to 2026-04-03


In [16]:
# Create forecast summary table
forecast_df = pd.DataFrame({
    "Trading Session": range(1, forecast_horizon + 1),
    "Forecast Date": future_dates,
    "Forecasted Close": [round(f, 2) for f in future_forecasts]
})

print("=" * 50)
print("       30-SESSION FUTURE FORECAST TABLE")
print("=" * 50)
print(forecast_df.to_string(index=False))

       30-SESSION FUTURE FORECAST TABLE
 Trading Session Forecast Date  Forecasted Close
               1    2026-02-23          25614.96
               2    2026-02-24          25627.20
               3    2026-02-25          25618.11
               4    2026-02-26          25622.94
               5    2026-02-27          25629.20
               6    2026-03-02          25634.15
               7    2026-03-03          25638.67
               8    2026-03-04          25643.76
               9    2026-03-05          25648.54
              10    2026-03-06          25653.40
              11    2026-03-09          25658.54
              12    2026-03-10          25663.88
              13    2026-03-11          25669.30
              14    2026-03-12          25674.62
              15    2026-03-13          25679.56
              16    2026-03-16          25684.69
              17    2026-03-17          25689.95
              18    2026-03-18          25695.34
              19    2026-03-1

In [17]:
# Forecast summary statistics
print("=" * 55)
print("       FORECAST SUMMARY STATISTICS")
print("=" * 55)
print(f"  Last Observed Close:     ₹{last_known_close:.2f}")
print(f"  Forecast Horizon:        {forecast_horizon} trading sessions")
print(f"  First Forecast Value:    ₹{future_forecasts[0]:.2f}")
print(f"  Last Forecast Value:     ₹{future_forecasts[-1]:.2f}")
print(f"  Minimum Forecast:        ₹{min(future_forecasts):.2f}")
print(f"  Maximum Forecast:        ₹{max(future_forecasts):.2f}")
forecast_change = future_forecasts[-1] - last_known_close
forecast_change_pct = (forecast_change / last_known_close) * 100
print(f"  Forecast Change:         ₹{forecast_change:.2f}")
print(f"  Forecast Change (%):     {forecast_change_pct:.4f}%")
print("=" * 55)

       FORECAST SUMMARY STATISTICS
  Last Observed Close:     ₹25571.25
  Forecast Horizon:        30 trading sessions
  First Forecast Value:    ₹25614.96
  Last Forecast Value:     ₹25756.65
  Minimum Forecast:        ₹25614.96
  Maximum Forecast:        ₹25756.65
  Forecast Change:         ₹185.40
  Forecast Change (%):     0.7250%


## 12. Forecast Visualization

We visualize the recent historical closing prices alongside the future forecasted values.

In [18]:
# Create Future Forecast chart
fig, ax = plt.subplots(figsize=(14, 6))

# Plot last 90 trading days of historical data for context
lookback = 90
hist_dates = pd.to_datetime(dates[-lookback:])
hist_close = close_series[-lookback:]

ax.plot(hist_dates, hist_close, label="Historical Close (Last 90 Days)", color="#1f77b4", linewidth=1.5)
ax.plot(future_dates, future_forecasts, label="Forecasted Close (30 Sessions)", color="#d62728", linewidth=2, linestyle="--", marker="o", markersize=3)

# Mark the boundary between historical and forecast
ax.axvline(x=last_date, color="gray", linestyle=":", alpha=0.7, label="Forecast Start")

ax.set_title("NIFTY 50 — Historical Close + 30-Session Future Forecast", fontsize=14, fontweight="bold")
ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Closing Price (₹)", fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("assets/week2_future_forecast.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: assets/week2_future_forecast.png")

Chart saved: assets/week2_future_forecast.png


## 13. Forecast Interpretation

In [19]:
# Detailed forecast interpretation
first_forecast = future_forecasts[0]
last_forecast = future_forecasts[-1]
forecast_change = last_forecast - last_known_close
forecast_change_pct = (forecast_change / last_known_close) * 100

if last_forecast > last_known_close:
    direction = "upward"
elif last_forecast < last_known_close:
    direction = "downward"
else:
    direction = "relatively stable"

print("FORECAST INTERPRETATION")
print("-" * 50)
print(f"\n1. GENERAL DIRECTION:")
print(f"   The model-based forecast suggests a {direction} path for NIFTY 50.")
print(f"\n2. FORECAST RANGE:")
print(f"   The forecasted values range from ₹{min(future_forecasts):.2f} to ₹{max(future_forecasts):.2f}.")
print(f"   This represents a range of approximately ₹{max(future_forecasts) - min(future_forecasts):.2f} points.")
print(f"\n3. CHANGE FROM LAST ACTUAL CLOSE:")
print(f"   Last actual close: ₹{last_known_close:.2f}")
print(f"   Last forecasted close: ₹{last_forecast:.2f}")
print(f"   Change: ₹{forecast_change:.2f} ({forecast_change_pct:.4f}%)")
print(f"\n4. MODEL PERFORMANCE ON TEST SET:")
print(f"   MAE = {mae:.2f}, RMSE = {rmse:.2f}, MAPE = {mape:.4f}%")
print(f"   The model showed {'strong' if r2 > 0.99 else 'good' if r2 > 0.95 else 'moderate'} predictive power on the test set (R² = {r2:.6f}).")
print(f"\n5. RELIABILITY ASSESSMENT:")
print(f"   This forecast should be considered INDICATIVE only.")
print(f"   It is a historical pattern-based estimate and does NOT guarantee future market performance.")
print(f"   Forecast uncertainty increases as the horizon extends further into the future.")

FORECAST INTERPRETATION
--------------------------------------------------

1. GENERAL DIRECTION:
   The model-based forecast suggests a upward path for NIFTY 50.

2. FORECAST RANGE:
   The forecasted values range from ₹25614.96 to ₹25756.65.
   This represents a range of approximately ₹141.69 points.

3. CHANGE FROM LAST ACTUAL CLOSE:
   Last actual close: ₹25571.25
   Last forecasted close: ₹25756.65
   Change: ₹185.40 (0.7250%)

4. MODEL PERFORMANCE ON TEST SET:
   MAE = 82.01, RMSE = 114.23, MAPE = 0.3747%
   The model showed strong predictive power on the test set (R² = 0.998622).

5. RELIABILITY ASSESSMENT:
   This forecast should be considered INDICATIVE only.
   It is a historical pattern-based estimate and does NOT guarantee future market performance.
   Forecast uncertainty increases as the horizon extends further into the future.


## 14. Limitations

It is important to acknowledge the limitations of this forecast model:

### Model Limitations
1. **Limited feature set:** The model uses only 7 historical price-based features. Real financial markets are influenced by many additional factors.
2. **No external variables:** The model does not incorporate macroeconomic conditions, interest rates, global market movements, political events, or company-level developments.
3. **Linear assumption:** Linear Regression assumes a linear relationship between features and the target, which may not capture all market dynamics.
4. **Recursive error accumulation:** In multi-step forecasting, prediction errors compound as each forecasted value is used as input for the next.

### Data Limitations
5. **Historical patterns do not guarantee future performance.** Financial markets can behave in ways that historical data does not predict.
6. **Market shocks and black swan events** (e.g., pandemics, geopolitical crises) are inherently unpredictable from historical price data alone.
7. **Forecast accuracy deteriorates** as the forecast horizon increases.

### Ethical Considerations
8. **This model should NOT be interpreted as financial advice.** It is an academic exercise for learning purposes only.
9. **No investment decisions** should be made based solely on this forecast.
10. **Past performance** of any model does not guarantee future predictive accuracy.

## 15. Conclusion

### Summary

In this Week 2 notebook, we successfully:

1. **Selected the NIFTY 50 Closing Price** as the financial indicator for forecasting, justified by its relevance and availability in the Week 1 dataset.

2. **Engineered 7 lag-based features** (Close_Lag_1, Close_Lag_5, Close_Lag_10, Close_Lag_20, MA_5, MA_20, Daily_Return_Pct) from purely historical data.

3. **Implemented a Linear Regression model** as a simple, interpretable, and reproducible baseline forecasting approach.

4. **Used an 80/20 chronological train/test split** to prevent data leakage and simulate real-world forecasting conditions.

5. **Evaluated the model** on the unseen test set, achieving strong predictive performance (MAE: 82.01, RMSE: 114.23, MAPE: 0.3747%).

6. **Generated a 30-session future forecast** (2026-02-23 to 2026-04-03) using recursive multi-step forecasting.

7. **Visualized results** through professional charts for both actual vs predicted and future forecast.

### Key Takeaway

The Linear Regression model using lag-based features provides a reasonable baseline for NIFTY 50 closing price forecasting. However, the forecast should be treated as **indicative only** and not as financial advice. Future work could incorporate additional features (volume, volatility, external indicators) and more sophisticated models to improve forecast accuracy.